# Notebook 06 sur GPU : référence et modèle sans variables protégées

Avant de lancer : **Exécution → Modifier le type d'exécution → GPU**, puis **Exécution → Tout exécuter**.

Dix entraînements (deux variantes × cinq plis, environ 75 min) et les valeurs de Shapley du pli 1. Les résultats sont écrits dans Google Drive (`MyDrive/hmda_repro_artifacts/06_interpretation`) : si la session est coupée, relancer tout, les plis terminés sont ignorés.

In [ ]:
# 1. Dépôt (branche interpretation), données et Drive
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/repo'
if not os.path.isdir(REPO):
    subprocess.check_call(['git', 'clone', '-q', 'https://github.com/xiao98/demandes-de-pret-immobilier-aux-etats-unis.git', REPO])
subprocess.check_call(['git', '-C', REPO, 'fetch', '-q', 'origin'])
subprocess.check_call(['git', '-C', REPO, 'checkout', '-q', '--detach', 'origin/interpretation'])
print('Commit :', subprocess.check_output(['git', '-C', REPO, 'log', '--oneline', '-1'], text=True).strip())
os.makedirs(f'{REPO}/data', exist_ok=True)
for name in ['train.csv', 'test.csv', 'sample_submission.csv']:      # les CSV sont dans l'historique du dépôt
    with open(f'{REPO}/data/{name}', 'wb') as handle:
        handle.write(subprocess.check_output(['git', '-C', REPO, 'show', f'c8b09fe:data/{name}']))
DRIVE_ARTIFACTS = '/content/drive/MyDrive/hmda_repro_artifacts'
os.makedirs(DRIVE_ARTIFACTS, exist_ok=True)
if not os.path.exists(f'{REPO}/artifacts'):
    os.symlink(DRIVE_ARTIFACTS, f'{REPO}/artifacts')
os.environ['HMDA_PROJECT_DIR'] = REPO
os.chdir(REPO)
subprocess.check_call(['python', 'tools/build_notebooks.py', '/content/gen'])
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, text=True, capture_output=True).stdout
assert gpu.strip(), "Pas de GPU : Exécution → Modifier le type d'exécution → GPU"
print('GPU :', gpu.strip())

In [ ]:
# 2. Notebooks 01 et 02 : données préparées, plis, variables (environ 2 min)
import nbformat

def run_notebook(name, replace=None):
    nb = nbformat.read(f'/content/gen/{name}', 4)
    space = {'__name__': '__main__'}
    for index, cell in enumerate(nb.cells):
        if cell.cell_type != 'code':
            continue
        source = cell.source
        if replace and replace[0] in source:
            source = source.replace(*replace)
        try:
            exec(compile(source, f'{name}[{index}]', 'exec'), space)
        except ValueError as error:
            if 'Aucun résultat' not in str(error):      # tableaux d'ablation de 02 : résultats absents, attendu
                raise
            print(f"[{name} cellule {index}] ignorée : résultats d'ablation absents")
    return space

run_notebook('01_data_exploration.ipynb')
run_notebook('02_feature_engineering.ipynb')

In [ ]:
# 3. Notebook 06 avec le bouton d'entraînement activé (environ 75 min), puis ses tableaux et figures
G = run_notebook('06_interpretation.ipynb', replace=("'protected_ablation': False", "'protected_ablation': True"))
assert G['TRAIN']['protected_ablation'], "Le bouton d'entraînement n'a pas été activé"

In [ ]:
# 4. Téléchargement des résultats
import shutil
shutil.make_archive('/content/interpretation', 'zip', G['OUTPUT_DIR'])
from google.colab import files
files.download('/content/interpretation.zip')